# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

# Import Functions
sys.path.append("../../")

from matplotlib import pyplot as plt
from matplotlib import pyplot as plt

from os.path import join, exists
import sys
sys.path.append("../") # Add directory containing src/data to path

from src.models.postnet.posterior_networks.run import run
from src.models.postnet.posterior_networks.run_eval import run_eval

from src.configs.blood_config import data_name, batch_size, eval_batch_size, \
    data_name_ood_in, data_name_ood_out, data_name_ood_diff, postnet_param, num_classes

from src.file_manager.filepath import FilePath
from src.models.postnet.posterior_networks.PosteriorNetwork import PosteriorNetwork
from src.configs.default_configs import fn_model, fn_pred
from src.models.postnet.result_processing import process_pn_results
from src.file_manager.load_save_df import load_pred_df, save_pred_perf_df
from src.evaluation.evaluate import get_model_performance
from src.evaluation.inference import split_test_set

from src.data_generator.blood import load_bloodmnist_data_dict
from src.data_generator.raabin import load_raabin_data_dict
from src.data_generator.bonemarrow import load_bonemarrow_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets

from cur_seed import seed
# seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_ood_in = FilePath(data_name=data_name_ood_in, seed=seed)
fp_ood_out = FilePath(data_name=data_name_ood_out, seed=seed)
fp_ood_diff = FilePath(data_name=data_name_ood_diff, seed=seed)

directory_model=fp.get_parent_folder(folder_name=fn_model)
directory_results=fp.get_parent_folder(folder_name=fn_pred)
fn_model = f"model-dpn-{seed}-{data_name}-{seed}-conv-[224, 224, 3]-{num_classes}"

# Load Data

In [ ]:
print("Loading Data Dict")
data_dict = load_bloodmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])

# Add ID Class OOD into the Test Set
print("Loading ID Class OOD Data Dict")
data_dict_ood_in = load_raabin_data_dict(fp_preprocessed=fp_ood_in.get_preprocessed_folder())
data_dict_ood_in = process_dataset_for_ood(data_dict, data_dict_ood_in, seed)

# ODD Class OOD
print("Loading OOD Class OOD Data Dict")
data_dict_ood_out = load_bonemarrow_data_dict(fp_preprocessed=fp_ood_out.get_preprocessed_folder())
data_dict_ood_out = process_dataset_for_ood(data_dict, data_dict_ood_out, seed)

# OOD Modality
print("Loading OOD Modality OOD Data Dict")
data_dict_ood_diff = load_chestmnist_data_dict(
    fp_preprocessed=fp_ood_diff.get_preprocessed_folder(), only_test=True, num_classes=8)
data_dict_ood_diff = process_dataset_for_ood(data_dict, data_dict_ood_diff, seed)

data_dict = left_join_datasets(data_dict, data_dict_ood_in)

# Training

In [ ]:
if not exists(join(directory_model, fn_model)):
    results_metrics = run(
        data_dict=data_dict,
        # Directory
        directory_model=directory_model,
        directory_results=directory_results,
        # Seeds
        seed_dataset=seed, # No shuffling
        seed_model=seed,
        **postnet_param
    )

# Prediction

In [ ]:
ood_postnet_param = postnet_param.copy()
ood_dicts = {
    "ood_in_raabin": data_dict_ood_in, 
    "ood_out_bonemarrow": data_dict_ood_out,
    "ood_chestmnist": data_dict_ood_diff}
fp_results = run_eval(
    data_dict=data_dict,
    ood_data_dicts = ood_dicts,
    # Directory
    directory_model=directory_model,
    directory_results=directory_results,
    # Seeds
    seed_dataset=seed, # No shuffling
    seed_model=seed,
    **ood_postnet_param,
    fn_model=fn_model,
    override=True
)

# Process Output

In [ ]:
ood_dicts = {
    "ood_in_raabin": data_dict_ood_in, 
    "ood_out_bonemarrow": data_dict_ood_out,
    "ood_chestmnist": data_dict_ood_diff
}
process_pn_results(data_dict, ood_dicts, directory_results, fp)

# Evaluate Pred Perf

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=PosteriorNetwork)
pred_df = split_test_set(
        pred_df, split_col="split", new_split_col="split_perf", 
        num_ori_test=num_ori_test, labels=["Test-Blood", "Test-Raabin"])
perf_df = get_model_performance(all_pred_df=pred_df, data_dict=data_dict, label="pn", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=PosteriorNetwork)
perf_df